In [6]:
from pathlib import Path
from typing import Literal, Optional

import gc

import joblib
import numpy as np
import pandas as pd

from scipy import sparse
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.preprocessing import normalize
from tqdm import tqdm


# ============================================================
# CONFIG
# ============================================================

PROCESSED_DIR = Path("../datasets/processed/PAN2011_300")
ARTIFACT_DIR = Path("../artifacts/tfidf_hashing")

SOURCE_CHUNKS_PATH = PROCESSED_DIR / "source_chunks_lsa_esa.parquet"
SUSPICIOUS_CHUNKS_PATH = PROCESSED_DIR / "suspicious_chunks_lsa_esa.parquet"
SOURCE_CANONICAL_CHUNKS_PATH = PROCESSED_DIR / "source_chunks.parquet"

SUSPICIOUS_DOC_ID = "part14__suspicious-document06510.txt"

OUTPUT_CANDIDATES_PATH = PROCESSED_DIR / "tfidf_candidates_suspicious_doc.parquet"
OUTPUT_TOP_DOCS_MAX_PATH = PROCESSED_DIR / "tfidf_top_source_documents_by_max_score.parquet"
OUTPUT_TOP_DOCS_MEAN_PATH = PROCESSED_DIR / "tfidf_top_source_documents_by_mean_score.parquet"

# Options:
# - "char": best for exact copy-paste and small edits
# - "word": good for word/phrase reuse
TFIDF_MODE: Literal["char", "word"] = "char"

# Change this to True only when rebuilding the TF-IDF source index.
BUILD_INDEX = True


# ============================================================
# MEMORY CLEANUP
# ============================================================

def cleanup_memory() -> None:
    gc.collect()


# ============================================================
# LOAD CHUNKS
# ============================================================

def load_tfidf_chunks(
    path: Path,
    text_column: str = "lsa_esa_text",
) -> pd.DataFrame:
    df = pd.read_parquet(path)

    required_columns = {
        "chunk_id",
        "doc_id",
        "chunk_index",
        "start_char",
        "end_char",
        text_column,
    }

    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns in {path}: {missing}")

    df = df.copy()
    df[text_column] = df[text_column].fillna("").astype(str)
    df = df[df[text_column].str.strip() != ""].reset_index(drop=True)

    return df


# ============================================================
# HASHING VECTORIZER
# ============================================================

def make_hashing_vectorizer(
    mode: Literal["char", "word"],
    n_features: int,
) -> HashingVectorizer:
    """
    HashingVectorizer avoids storing a huge vocabulary.

    Important:
    - alternate_sign=False keeps all values positive.
    - norm=None because we apply TF-IDF and L2 normalization manually.
    - dtype=np.float32 reduces memory compared to float64.
    """

    if mode == "char":
        return HashingVectorizer(
            analyzer="char_wb",
            ngram_range=(5, 7),
            n_features=n_features,
            lowercase=True,
            strip_accents="unicode",
            alternate_sign=False,
            norm=None,
            binary=False,
            dtype=np.float32,
        )

    if mode == "word":
        return HashingVectorizer(
            analyzer="word",
            ngram_range=(1, 3),
            n_features=n_features,
            lowercase=True,
            strip_accents="unicode",
            alternate_sign=False,
            norm=None,
            binary=False,
            dtype=np.float32,
        )

    raise ValueError(f"Unsupported TF-IDF mode: {mode}")


# ============================================================
# TF-IDF TRANSFORM HELPERS
# ============================================================

def compute_hashed_document_frequencies(
    texts: list[str],
    vectorizer: HashingVectorizer,
    n_features: int,
    batch_size: int,
) -> tuple[np.ndarray, int]:
    """
    First pass.

    Computes document frequency per hashed feature without storing the full matrix.
    """

    df_counts = np.zeros(n_features, dtype=np.int64)
    total_docs = 0

    total_batches = (len(texts) + batch_size - 1) // batch_size

    print("Computing hashed document frequencies...")

    for start in tqdm(
        range(0, len(texts), batch_size),
        total=total_batches,
        desc="DF batches",
    ):
        end = min(start + batch_size, len(texts))
        batch_texts = texts[start:end]

        counts = vectorizer.transform(batch_texts).tocsr()

        # Convert term-frequency counts into binary document presence.
        counts.data[:] = 1.0

        # Sum feature presence across documents in this batch.
        batch_df = np.asarray(counts.sum(axis=0)).ravel().astype(np.int64)

        df_counts += batch_df
        total_docs += len(batch_texts)

        del counts
        del batch_df

        if total_docs % (batch_size * 50) == 0:
            cleanup_memory()

    return df_counts, total_docs


def compute_sklearn_style_idf(
    df_counts: np.ndarray,
    total_docs: int,
) -> np.ndarray:
    """
    Smooth IDF compatible with sklearn's default formula:

    idf = log((1 + n_docs) / (1 + df)) + 1
    """

    idf = np.log((1.0 + total_docs) / (1.0 + df_counts.astype(np.float32))) + 1.0
    return idf.astype(np.float32)


def transform_texts_to_tfidf(
    texts: list[str],
    vectorizer: HashingVectorizer,
    idf: np.ndarray,
) -> sparse.csr_matrix:
    """
    Transform texts into L2-normalized hashed TF-IDF vectors.
    """

    counts = vectorizer.transform(texts).tocsr().astype(np.float32)

    # Apply sublinear TF: 1 + log(tf)
    if counts.nnz > 0:
        counts.data = 1.0 + np.log(counts.data)

    tfidf = counts.multiply(idf).tocsr().astype(np.float32)

    # L2 normalization makes dot product = cosine similarity.
    tfidf = normalize(tfidf, norm="l2", axis=1, copy=False)

    return tfidf.astype(np.float32)


# ============================================================
# BUILD SHARDED HASHED TF-IDF INDEX
# ============================================================

def build_tfidf_index(
    source_chunks_path: Path,
    artifact_dir: Path,
    text_column: str = "lsa_esa_text",
    mode: Literal["char", "word"] = "char",
    n_features: int = 2**20,
    batch_size: int = 2048,
    shard_size_rows: int = 100_000,
    max_source_chunks: Optional[int] = None,
) -> None:
    """
    Build and save a sharded hashed TF-IDF source index.

    This is designed for very large source collections.

    Creates:
    - tfidf_vectorizer.joblib
    - tfidf_idf.npy
    - source_tfidf_metadata.parquet
    - tfidf_shards.parquet
    - shards/source_tfidf_shard_00000.npz
    - shards/source_tfidf_shard_00001.npz
    - tfidf_config.joblib
    """

    artifact_dir = Path(artifact_dir) / mode
    shard_dir = artifact_dir / "shards"
    shard_dir.mkdir(parents=True, exist_ok=True)

    source_df = load_tfidf_chunks(
        path=source_chunks_path,
        text_column=text_column,
    )

    if max_source_chunks is not None:
        source_df = source_df.head(max_source_chunks).reset_index(drop=True)

    print(f"Loaded {len(source_df)} source chunks")
    print(f"Building HASHED TF-IDF index with mode: {mode}")
    print(f"Hash features: {n_features}")
    print(f"Batch size: {batch_size}")
    print(f"Shard size rows: {shard_size_rows}")

    metadata_columns = [
        "chunk_id",
        "doc_id",
        "chunk_index",
        "start_char",
        "end_char",
    ]

    optional_columns = ["file_name", "relative_path", "part", "word_count"]
    metadata_columns += [col for col in optional_columns if col in source_df.columns]

    source_metadata = source_df[metadata_columns].copy()
    source_texts = source_df[text_column].tolist()

    del source_df
    cleanup_memory()

    vectorizer = make_hashing_vectorizer(
        mode=mode,
        n_features=n_features,
    )

    df_counts, total_docs = compute_hashed_document_frequencies(
        texts=source_texts,
        vectorizer=vectorizer,
        n_features=n_features,
        batch_size=batch_size,
    )

    idf = compute_sklearn_style_idf(
        df_counts=df_counts,
        total_docs=total_docs,
    )

    print("Finished computing IDF")
    print(f"Total documents/chunks: {total_docs}")

    del df_counts
    cleanup_memory()

    print("Building and saving TF-IDF shards...")

    shard_infos = []
    shard_id = 0
    shard_start_row = 0

    current_parts = []
    current_rows = 0

    total_batches = (len(source_texts) + batch_size - 1) // batch_size

    for start in tqdm(
        range(0, len(source_texts), batch_size),
        total=total_batches,
        desc="TF-IDF shard batches",
    ):
        end = min(start + batch_size, len(source_texts))
        batch_texts = source_texts[start:end]

        batch_tfidf = transform_texts_to_tfidf(
            texts=batch_texts,
            vectorizer=vectorizer,
            idf=idf,
        )

        current_parts.append(batch_tfidf)
        current_rows += batch_tfidf.shape[0]

        should_save_shard = current_rows >= shard_size_rows
        is_last_batch = end == len(source_texts)

        if should_save_shard or is_last_batch:
            shard_matrix = sparse.vstack(current_parts, format="csr").astype(np.float32)

            shard_path = shard_dir / f"source_tfidf_shard_{shard_id:05d}.npz"

            print(
                f"\nSaving TF-IDF shard {shard_id} | "
                f"rows={shard_matrix.shape[0]} | "
                f"nnz={shard_matrix.nnz} | "
                f"path={shard_path}"
            )

            sparse.save_npz(shard_path, shard_matrix, compressed=True)

            shard_infos.append({
                "shard_id": shard_id,
                "shard_path": str(shard_path),
                "start_row": shard_start_row,
                "num_rows": shard_matrix.shape[0],
                "end_row": shard_start_row + shard_matrix.shape[0],
                "num_features": shard_matrix.shape[1],
                "nnz": shard_matrix.nnz,
            })

            shard_start_row += shard_matrix.shape[0]
            shard_id += 1

            del shard_matrix
            del current_parts

            current_parts = []
            current_rows = 0

            cleanup_memory()

    del source_texts
    cleanup_memory()

    total_rows = sum(info["num_rows"] for info in shard_infos)

    if len(source_metadata) != total_rows:
        raise ValueError(
            f"Metadata/shard mismatch: metadata rows={len(source_metadata)}, "
            f"TF-IDF rows={total_rows}"
        )

    print("Saving TF-IDF artifacts...")

    joblib.dump(vectorizer, artifact_dir / "tfidf_vectorizer.joblib")
    np.save(artifact_dir / "tfidf_idf.npy", idf)

    source_metadata.to_parquet(
        artifact_dir / "source_tfidf_metadata.parquet",
        index=False,
    )

    shard_info_df = pd.DataFrame(shard_infos)

    shard_info_df.to_parquet(
        artifact_dir / "tfidf_shards.parquet",
        index=False,
    )

    config = {
        "mode": mode,
        "text_column": text_column,
        "n_features": n_features,
        "batch_size": batch_size,
        "shard_size_rows": shard_size_rows,
        "max_source_chunks": max_source_chunks,
        "source_chunks_path": str(source_chunks_path),
        "num_shards": len(shard_infos),
        "total_rows": total_rows,
        "similarity": "cosine_similarity_via_l2_normalized_dot_product",
        "vectorizer": "HashingVectorizer",
        "idf_formula": "log((1 + n_docs) / (1 + df)) + 1",
    }

    joblib.dump(config, artifact_dir / "tfidf_config.joblib")

    cleanup_memory()

    print(f"Saved TF-IDF artifacts to: {artifact_dir}")
    print(f"Total shards: {len(shard_infos)}")
    print(f"Total rows: {total_rows}")


# ============================================================
# LOAD SHARDED TF-IDF INDEX
# ============================================================

def load_tfidf_index(
    artifact_dir: Path,
    mode: Literal["char", "word"] = "char",
):
    artifact_dir = Path(artifact_dir) / mode

    vectorizer_path = artifact_dir / "tfidf_vectorizer.joblib"
    idf_path = artifact_dir / "tfidf_idf.npy"
    metadata_path = artifact_dir / "source_tfidf_metadata.parquet"
    shards_path = artifact_dir / "tfidf_shards.parquet"
    config_path = artifact_dir / "tfidf_config.joblib"

    for path in [vectorizer_path, idf_path, metadata_path, shards_path, config_path]:
        if not path.exists():
            raise FileNotFoundError(f"Missing TF-IDF artifact: {path}")

    print("Loading sharded TF-IDF artifacts...")

    vectorizer = joblib.load(vectorizer_path)
    idf = np.load(idf_path).astype(np.float32)
    source_metadata = pd.read_parquet(metadata_path)
    shard_info_df = pd.read_parquet(shards_path)
    config = joblib.load(config_path)

    expected_rows = int(shard_info_df["num_rows"].sum())

    if len(source_metadata) != expected_rows:
        raise ValueError(
            f"Metadata/shard mismatch: metadata rows={len(source_metadata)}, "
            f"TF-IDF rows={expected_rows}"
        )

    print(f"Loaded source metadata rows: {len(source_metadata)}")
    print(f"Loaded TF-IDF shards: {len(shard_info_df)}")

    return vectorizer, idf, shard_info_df, source_metadata, config


# ============================================================
# SEARCH SHARDED TF-IDF
# ============================================================

def search_tfidf_shards(
    query_tfidf: sparse.csr_matrix,
    shard_info_df: pd.DataFrame,
    top_k: int,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Search all TF-IDF shards and return global top-k source rows per query row.

    Returns:
    - final_scores: shape (num_queries, top_k)
    - final_indices: global source metadata row indices, shape (num_queries, top_k)
    """

    num_queries = query_tfidf.shape[0]

    per_query_scores = [[] for _ in range(num_queries)]
    per_query_indices = [[] for _ in range(num_queries)]

    for shard_row in tqdm(
        shard_info_df.itertuples(index=False),
        total=len(shard_info_df),
        desc="Searching TF-IDF shards",
    ):
        shard_path = Path(shard_row.shard_path)
        shard_start_row = int(shard_row.start_row)

        if not shard_path.exists():
            raise FileNotFoundError(f"Missing TF-IDF shard: {shard_path}")

        source_tfidf_shard = sparse.load_npz(shard_path).tocsr()

        # Sparse cosine similarity.
        # Since both matrices are L2-normalized, dot product = cosine similarity.
        similarities = query_tfidf @ source_tfidf_shard.T

        for query_i in range(num_queries):
            sims_sparse = similarities.getrow(query_i)

            if sims_sparse.nnz == 0:
                continue

            candidate_indices = sims_sparse.indices
            candidate_scores = sims_sparse.data

            safe_top_k = min(top_k, len(candidate_scores))

            top_local_indices = np.argpartition(
                -candidate_scores,
                safe_top_k - 1,
            )[:safe_top_k]

            top_local_indices = top_local_indices[
                np.argsort(-candidate_scores[top_local_indices])
            ]

            global_indices = candidate_indices[top_local_indices] + shard_start_row
            top_scores = candidate_scores[top_local_indices]

            per_query_indices[query_i].extend(global_indices.tolist())
            per_query_scores[query_i].extend(top_scores.tolist())

        del source_tfidf_shard
        del similarities
        cleanup_memory()

    final_scores = np.full((num_queries, top_k), -np.inf, dtype=np.float32)
    final_indices = np.full((num_queries, top_k), -1, dtype=np.int64)

    for query_i in range(num_queries):
        scores = np.asarray(per_query_scores[query_i], dtype=np.float32)
        indices = np.asarray(per_query_indices[query_i], dtype=np.int64)

        if len(scores) == 0:
            continue

        safe_top_k = min(top_k, len(scores))

        top_positions = np.argpartition(
            -scores,
            safe_top_k - 1,
        )[:safe_top_k]

        top_positions = top_positions[np.argsort(-scores[top_positions])]

        final_scores[query_i, :safe_top_k] = scores[top_positions]
        final_indices[query_i, :safe_top_k] = indices[top_positions]

    return final_scores, final_indices


# ============================================================
# QUERY ONE SUSPICIOUS DOCUMENT
# ============================================================

def retrieve_tfidf_candidates_for_suspicious_doc(
    suspicious_chunks_path: Path,
    artifact_dir: Path,
    suspicious_doc_id: str,
    output_path: Path,
    text_column: str = "lsa_esa_text",
    mode: Literal["char", "word"] = "char",
    top_k: int = 100,
    batch_size: int = 8,
    max_suspicious_chunks: Optional[int] = None,
) -> pd.DataFrame:
    """
    Query the sharded hashed TF-IDF source index using one selected suspicious document.

    Returns top-k source chunks for each suspicious chunk.
    """

    vectorizer, idf, shard_info_df, source_metadata, config = load_tfidf_index(
        artifact_dir=artifact_dir,
        mode=mode,
    )

    suspicious_df = load_tfidf_chunks(
        path=suspicious_chunks_path,
        text_column=text_column,
    )

    suspicious_df = suspicious_df[
        suspicious_df["doc_id"] == suspicious_doc_id
    ].copy()

    if suspicious_df.empty:
        raise ValueError(f"No suspicious chunks found for doc_id: {suspicious_doc_id}")

    if max_suspicious_chunks is not None:
        suspicious_df = suspicious_df.head(max_suspicious_chunks).reset_index(drop=True)

    print(f"Selected suspicious document: {suspicious_doc_id}")
    print(f"Suspicious chunks to query: {len(suspicious_df)}")
    print(f"Searching global top-{top_k} source chunks per suspicious chunk")
    print(f"TF-IDF mode: {mode}")

    results = []

    for start in range(0, len(suspicious_df), batch_size):
        end = min(start + batch_size, len(suspicious_df))
        batch_df = suspicious_df.iloc[start:end]

        batch_texts = batch_df[text_column].tolist()

        suspicious_tfidf = transform_texts_to_tfidf(
            texts=batch_texts,
            vectorizer=vectorizer,
            idf=idf,
        )

        scores, indices = search_tfidf_shards(
            query_tfidf=suspicious_tfidf,
            shard_info_df=shard_info_df,
            top_k=top_k,
        )

        for local_i, suspicious_row in enumerate(batch_df.itertuples(index=False)):
            for rank in range(top_k):
                source_idx = int(indices[local_i, rank])
                score = float(scores[local_i, rank])

                if source_idx < 0 or not np.isfinite(score):
                    continue

                source_row = source_metadata.iloc[source_idx]

                results.append({
                    "suspicious_chunk_id": suspicious_row.chunk_id,
                    "suspicious_doc_id": suspicious_row.doc_id,
                    "suspicious_chunk_index": suspicious_row.chunk_index,
                    "suspicious_start_char": suspicious_row.start_char,
                    "suspicious_end_char": suspicious_row.end_char,

                    "source_chunk_id": source_row["chunk_id"],
                    "source_doc_id": source_row["doc_id"],
                    "source_chunk_index": source_row["chunk_index"],
                    "source_start_char": source_row["start_char"],
                    "source_end_char": source_row["end_char"],

                    "TFIDF_score": score,
                    "TFIDF_rank": rank + 1,
                    "TFIDF_mode": mode,
                    "TFIDF_vectorizer": "hashing",
                })

        del suspicious_tfidf
        del scores
        del indices
        cleanup_memory()

        print(f"Processed suspicious chunks {start} to {end}")

    output_df = pd.DataFrame(results)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    output_path = output_path.with_name(
        output_path.stem + f"_{mode}_hashing" + output_path.suffix
    )

    output_df.to_parquet(output_path, index=False)

    print(f"Saved TF-IDF candidates to: {output_path}")
    print(f"Candidate rows: {len(output_df)}")

    return output_df


# ============================================================
# DOCUMENT-LEVEL MAX SCORE
# ============================================================

def get_top_source_documents_by_max_tfidf_score(
    candidates_df: pd.DataFrame,
    source_chunks_path: Path,
    suspicious_doc_id: str,
    top_n: int = 20,
    min_match_count: int = 1,
    output_path: Optional[Path] = None,
) -> pd.DataFrame:
    """
    Rank unique source documents by strongest TF-IDF chunk match.

    Best for copy-paste retrieval because plagiarism is often local.
    """

    required_columns = {
        "suspicious_doc_id",
        "suspicious_chunk_id",
        "source_doc_id",
        "source_chunk_id",
        "TFIDF_score",
    }

    missing = required_columns - set(candidates_df.columns)
    if missing:
        raise ValueError(f"Missing columns in candidates_df: {missing}")

    filtered_df = candidates_df[
        candidates_df["suspicious_doc_id"] == suspicious_doc_id
    ].copy()

    if filtered_df.empty:
        raise ValueError(f"No candidates found for suspicious_doc_id: {suspicious_doc_id}")

    grouped_df = (
        filtered_df
        .groupby("source_doc_id")
        .agg(
            max_TFIDF_score=("TFIDF_score", "max"),
            mean_TFIDF_score=("TFIDF_score", "mean"),
            min_TFIDF_score=("TFIDF_score", "min"),
            match_count=("TFIDF_score", "count"),
            unique_source_chunks=("source_chunk_id", "nunique"),
            unique_suspicious_chunks=("suspicious_chunk_id", "nunique"),
        )
        .reset_index()
    )

    grouped_df = grouped_df[grouped_df["match_count"] >= min_match_count].copy()

    grouped_df = (
        grouped_df
        .sort_values(
            [
                "max_TFIDF_score",
                "unique_suspicious_chunks",
                "match_count",
                "mean_TFIDF_score",
            ],
            ascending=[False, False, False, False],
        )
        .head(top_n)
        .reset_index(drop=True)
    )

    grouped_df["source_doc_rank"] = range(1, len(grouped_df) + 1)

    grouped_df = grouped_df[
        [
            "source_doc_rank",
            "source_doc_id",
            "max_TFIDF_score",
            "mean_TFIDF_score",
            "min_TFIDF_score",
            "match_count",
            "unique_source_chunks",
            "unique_suspicious_chunks",
        ]
    ]

    source_meta_df = pd.read_parquet(
        source_chunks_path,
        columns=["doc_id", "relative_path"],
    ).drop_duplicates("doc_id")

    source_meta_df = source_meta_df.rename(columns={
        "doc_id": "source_doc_id",
        "relative_path": "source_relative_path",
    })

    grouped_df = grouped_df.merge(
        source_meta_df,
        on="source_doc_id",
        how="left",
    )

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)

        mode = candidates_df["TFIDF_mode"].iloc[0] if "TFIDF_mode" in candidates_df.columns else "unknown"
        output_path = output_path.with_name(
            output_path.stem + f"_{mode}_hashing" + output_path.suffix
        )

        grouped_df.to_parquet(output_path, index=False)
        print(f"Saved top source documents to: {output_path}")

    return grouped_df


# ============================================================
# DOCUMENT-LEVEL MEAN SCORE
# ============================================================

def get_top_source_documents_by_mean_tfidf_score(
    candidates_df: pd.DataFrame,
    source_chunks_path: Path,
    suspicious_doc_id: str,
    top_n: int = 20,
    min_match_count: int = 4,
    output_path: Optional[Path] = None,
) -> pd.DataFrame:
    """
    Rank unique source documents by mean TF-IDF score.
    """

    required_columns = {
        "suspicious_doc_id",
        "suspicious_chunk_id",
        "source_doc_id",
        "source_chunk_id",
        "TFIDF_score",
    }

    missing = required_columns - set(candidates_df.columns)
    if missing:
        raise ValueError(f"Missing columns in candidates_df: {missing}")

    filtered_df = candidates_df[
        candidates_df["suspicious_doc_id"] == suspicious_doc_id
    ].copy()

    if filtered_df.empty:
        raise ValueError(f"No candidates found for suspicious_doc_id: {suspicious_doc_id}")

    grouped_df = (
        filtered_df
        .groupby("source_doc_id")
        .agg(
            mean_TFIDF_score=("TFIDF_score", "mean"),
            max_TFIDF_score=("TFIDF_score", "max"),
            min_TFIDF_score=("TFIDF_score", "min"),
            match_count=("TFIDF_score", "count"),
            unique_source_chunks=("source_chunk_id", "nunique"),
            unique_suspicious_chunks=("suspicious_chunk_id", "nunique"),
        )
        .reset_index()
    )

    grouped_df = grouped_df[grouped_df["match_count"] >= min_match_count].copy()

    grouped_df = (
        grouped_df
        .sort_values(
            ["mean_TFIDF_score", "match_count", "max_TFIDF_score"],
            ascending=[False, False, False],
        )
        .head(top_n)
        .reset_index(drop=True)
    )

    grouped_df["source_doc_rank"] = range(1, len(grouped_df) + 1)

    grouped_df = grouped_df[
        [
            "source_doc_rank",
            "source_doc_id",
            "mean_TFIDF_score",
            "max_TFIDF_score",
            "min_TFIDF_score",
            "match_count",
            "unique_source_chunks",
            "unique_suspicious_chunks",
        ]
    ]

    source_meta_df = pd.read_parquet(
        source_chunks_path,
        columns=["doc_id", "relative_path"],
    ).drop_duplicates("doc_id")

    source_meta_df = source_meta_df.rename(columns={
        "doc_id": "source_doc_id",
        "relative_path": "source_relative_path",
    })

    grouped_df = grouped_df.merge(
        source_meta_df,
        on="source_doc_id",
        how="left",
    )

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)

        mode = candidates_df["TFIDF_mode"].iloc[0] if "TFIDF_mode" in candidates_df.columns else "unknown"
        output_path = output_path.with_name(
            output_path.stem + f"_{mode}_hashing" + output_path.suffix
        )

        grouped_df.to_parquet(output_path, index=False)
        print(f"Saved top source documents to: {output_path}")

    return grouped_df


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    BUILD_INDEX = False
    TFIDF_MODE = "char"

    SUSPICIOUS_DOC_ID = "part14__suspicious-document06510.txt"

    if BUILD_INDEX:
        build_tfidf_index(
            source_chunks_path=SOURCE_CHUNKS_PATH,
            artifact_dir=ARTIFACT_DIR,
            text_column="lsa_esa_text",
            mode=TFIDF_MODE,
            n_features=2**20,
            batch_size=2048,
            shard_size_rows=100_000,
            max_source_chunks=None,
        )

    candidates_df = retrieve_tfidf_candidates_for_suspicious_doc(
        suspicious_chunks_path=SUSPICIOUS_CHUNKS_PATH,
        artifact_dir=ARTIFACT_DIR,
        suspicious_doc_id=SUSPICIOUS_DOC_ID,
        output_path=OUTPUT_CANDIDATES_PATH,
        text_column="lsa_esa_text",
        mode=TFIDF_MODE,
        top_k=500,
        batch_size=8,
        max_suspicious_chunks=None,
    )

    top_sources_max_df = get_top_source_documents_by_max_tfidf_score(
        candidates_df=candidates_df,
        source_chunks_path=SOURCE_CANONICAL_CHUNKS_PATH,
        suspicious_doc_id=SUSPICIOUS_DOC_ID,
        top_n=50,
        min_match_count=1,
        output_path=OUTPUT_TOP_DOCS_MAX_PATH,
    )

    top_sources_mean_df = get_top_source_documents_by_mean_tfidf_score(
        candidates_df=candidates_df,
        source_chunks_path=SOURCE_CANONICAL_CHUNKS_PATH,
        suspicious_doc_id=SUSPICIOUS_DOC_ID,
        top_n=50,
        min_match_count=1,
        output_path=OUTPUT_TOP_DOCS_MEAN_PATH,
    )

Loading sharded TF-IDF artifacts...
Loaded source metadata rows: 1900110
Loaded TF-IDF shards: 19
Selected suspicious document: part14__suspicious-document06510.txt
Suspicious chunks to query: 303
Searching global top-500 source chunks per suspicious chunk
TF-IDF mode: char


Searching TF-IDF shards: 100%|██████████| 19/19 [02:08<00:00,  6.78s/it]


Processed suspicious chunks 0 to 8


Searching TF-IDF shards: 100%|██████████| 19/19 [02:02<00:00,  6.45s/it]


Processed suspicious chunks 8 to 16


...

Processed suspicious chunks 280 to 288


Searching TF-IDF shards: 100%|██████████| 19/19 [02:02<00:00,  6.47s/it]


Processed suspicious chunks 288 to 296


Searching TF-IDF shards: 100%|██████████| 19/19 [01:58<00:00,  6.26s/it]


Processed suspicious chunks 296 to 303
Saved TF-IDF candidates to: ..\datasets\processed\PAN2011_300\tfidf_candidates_suspicious_doc_char_hashing.parquet
Candidate rows: 151500
Saved top source documents to: ..\datasets\processed\PAN2011_300\tfidf_top_source_documents_by_max_score_char_hashing.parquet
Saved top source documents to: ..\datasets\processed\PAN2011_300\tfidf_top_source_documents_by_mean_score_char_hashing.parquet


In [9]:
top_sources_max_df.head(10)

,source_doc_rank,source_doc_id,max_TFIDF_score,mean_TFIDF_score,min_TFIDF_score,match_count,unique_source_chunks,unique_suspicious_chunks,source_relative_path
0,1,part14__source-document06533.txt,0.593026,0.139629,0.064303,2371,358,218,part14/source-document06533.txt
1,2,part21__source-document10319.txt,0.433443,0.128328,0.062523,1232,193,67,part21/source-document10319.txt
2,3,part7__source-document03250.txt,0.378204,0.116180,0.063384,450,165,108,part7/source-document03250.txt
3,4,part21__source-document10244.txt,0.358823,0.126551,0.063370,165,36,24,part21/source-document10244.txt
4,5,part18__source-document08679.txt,0.339001,0.123414,0.068589,1551,303,59,part18/source-document08679.txt
5,6,part22__source-document10678.txt,0.265733,0.139196,0.074002,54,17,17,part22/source-document10678.txt
6,7,part4__source-document01797.txt,0.264369,0.122319,0.071811,129,29,27,part4/source-document01797.txt
7,8,part21__source-document10251.txt,0.262875,0.127325,0.075395,175,35,35,part21/source-document10251.txt
8,9,part4__source-document01907.txt,0.258463,0.119757,0.070863,190,33,44,part4/source-document01907.txt
9,10,part22__source-document10663.txt,0.256285,0.109993,0.067864,95,73,42,part22/source-document10663.txt
